In [ ]:
# ==========================================================
# Section 1: Project Configuration & Dataset Loading
# ==========================================================
#
# Purpose
# -------
# Initialise the project environment and load the datasets
# required for AfriHate preparation.
#
# This section:
#   • Configures project directories
#   • Sets reproducibility parameters
#   • Configures logging
#   • Loads the frozen PolitikWeli datasets
#   • Downloads the AfriHate Swahili dataset
#   • Performs initial schema validation
#
# Inputs
# ------
# data/processed/
#     politikweli_master.csv
#     politikweli_train.csv
#     politikweli_validation.csv
#     politikweli_test.csv
#
# Outputs
# -------
# In-memory DataFrames for subsequent preprocessing.
#
# ==========================================================

from pathlib import Path
import logging
import random

import numpy as np
import pandas as pd

from datasets import load_dataset

# ==========================================================
# Project Configuration
# ==========================================================

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [

    DATA_DIR,

    RAW_DIR,

    PROCESSED_DIR,

    REPORTS_DIR,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ==========================================================
# Reproducibility
# ==========================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

# ==========================================================
# Logging
# ==========================================================

logging.basicConfig(

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s"

)

logger = logging.getLogger(__name__)

logger.info("=" * 70)
logger.info("AFRIHATE DATA PREPARATION")
logger.info("=" * 70)

# ==========================================================
# Verify PolitikWeli Processed Files
# ==========================================================

required_files = {

    "master": PROCESSED_DIR / "politikweli_master.csv",

    "train": PROCESSED_DIR / "politikweli_train.csv",

    "validation": PROCESSED_DIR / "politikweli_validation.csv",

    "test": PROCESSED_DIR / "politikweli_test.csv",

}

logger.info("Checking PolitikWeli processed datasets...")

missing_files = [

    path

    for path in required_files.values()

    if not path.exists()

]

if missing_files:

    raise FileNotFoundError(

        "The following processed PolitikWeli datasets "
        "could not be found:\n\n"

        + "\n".join(str(file) for file in missing_files)

        + "\n\nRun 01_politikweli_prepare.ipynb first."

    )

logger.info("✓ All required PolitikWeli datasets found.")

# ==========================================================
# Load Frozen PolitikWeli Datasets
# ==========================================================

logger.info("Loading frozen PolitikWeli datasets...")

politikweli_master = pd.read_csv(

    required_files["master"],

    dtype={"text_id": str}

)

politikweli_train = pd.read_csv(

    required_files["train"],

    dtype={"text_id": str}

)

politikweli_validation = pd.read_csv(

    required_files["validation"],

    dtype={"text_id": str}

)

politikweli_test = pd.read_csv(

    required_files["test"],

    dtype={"text_id": str}

)

logger.info(
    f"Master dataset:      {len(politikweli_master):,}"
)

logger.info(
    f"Training dataset:    {len(politikweli_train):,}"
)

logger.info(
    f"Validation dataset:  {len(politikweli_validation):,}"
)

logger.info(
    f"Test dataset:        {len(politikweli_test):,}"
)

# ==========================================================
# Download AfriHate Dataset
# ==========================================================

logger.info("=" * 70)
logger.info("LOADING AFRIHATE DATASET")
logger.info("=" * 70)

logger.info(
    "Downloading AfriHate dataset from Hugging Face..."
)

afrihate_dataset = load_dataset(

    "afrihate/afrihate",

    "sw"

)

logger.info("✓ AfriHate dataset loaded successfully.")

logger.info("\nAvailable splits:")

for split in afrihate_dataset.keys():

    logger.info(

        f"  • {split}: "

        f"{len(afrihate_dataset[split]):,} samples"

    )

# ==========================================================
# Merge Original Splits
# ==========================================================

logger.info("Combining original Hugging Face splits...")

split_frames = []

for split_name in afrihate_dataset.keys():

    split_df = afrihate_dataset[split_name].to_pandas()

    split_df["hf_split"] = split_name

    split_frames.append(split_df)

afrihate_master = pd.concat(

    split_frames,

    ignore_index=True

)

logger.info(

    f"Combined dataset size: "

    f"{len(afrihate_master):,} samples"

)

# ==========================================================
# Initial Schema Validation
# ==========================================================

logger.info("=" * 70)
logger.info("VALIDATING DATASET SCHEMA")
logger.info("=" * 70)

expected_columns = {

    "text",

    "label"

}

missing_columns = expected_columns.difference(

    afrihate_master.columns

)

if missing_columns:

    raise ValueError(

        "The following required columns are missing:\n"

        f"{sorted(missing_columns)}"

    )

logger.info("✓ Required columns present.")

logger.info("\nDataset columns:")

for column in afrihate_master.columns:

    logger.info(f"  • {column}")

# ==========================================================
# Initial Missing Value Audit
# ==========================================================

logger.info("=" * 70)
logger.info("INITIAL DATA QUALITY AUDIT")
logger.info("=" * 70)

missing_summary = (

    afrihate_master

    .isnull()

    .sum()

    .sort_values(ascending=False)

)

print("\nMissing Values")

print(missing_summary)

logger.info(
    f"\nTotal records: {len(afrihate_master):,}"
)

logger.info(
    "Section 1 completed successfully."
)

In [ ]:
# ==========================================================
# Section 2: Data Cleaning & Initial Preprocessing
# ==========================================================
#
# Purpose
# -------
# Clean and standardise the AfriHate dataset prior to label
# harmonisation and cross-dataset validation.
#
# This section:
#   • Removes incomplete records
#   • Cleans tweet text
#   • Removes empty texts after cleaning
#   • Computes basic sequence statistics
#   • Displays sample preprocessing results
#
# Output
# ------
# afrihate_master (cleaned)
#
# ==========================================================

import re

logger.info("=" * 70)
logger.info("DATA CLEANING & PREPROCESSING")
logger.info("=" * 70)

# ==========================================================
# Text Cleaning Function
# ==========================================================

def clean_codeswitched_text(text):
    """
    Clean social media text while preserving linguistic
    characteristics useful for transformer models.

    Preserved:
        • Swahili-English code-switching
        • Hashtags
        • Emojis
        • Slang
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove zero-width characters
    text = re.sub(
        r'[\u200b\u200c\u200d\ufeff]',
        '',
        text
    )

    # Remove retweet prefix
    text = re.sub(
        r'^RT\s+@\w+:\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove URLs
    text = re.sub(
        r'http\S+|www\S+|https\S+',
        '',
        text,
        flags=re.MULTILINE
    )

    # Standardise user mentions
    text = re.sub(
        r'@\w+',
        '@USER',
        text
    )

    # Collapse whitespace
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

# ==========================================================
# Remove Missing Records
# ==========================================================

logger.info("Removing incomplete records...")

initial_size = len(afrihate_master)

afrihate_master = afrihate_master.dropna(
    subset=["text", "label"]
).copy()

removed = initial_size - len(afrihate_master)

logger.info(
    f"Removed {removed:,} incomplete records."
)

logger.info(
    f"Remaining records: {len(afrihate_master):,}"
)

# ==========================================================
# Clean Tweet Text
# ==========================================================

logger.info("Cleaning tweet text...")

afrihate_master["clean_text"] = (

    afrihate_master["text"]

    .apply(clean_codeswitched_text)

)

# ==========================================================
# Remove Empty Texts
# ==========================================================

before = len(afrihate_master)

afrihate_master = (

    afrihate_master[
        afrihate_master["clean_text"] != ""
    ]

    .copy()

)

removed = before - len(afrihate_master)

logger.info(
    f"Removed {removed:,} empty texts."
)

logger.info(
    f"Remaining records: {len(afrihate_master):,}"
)

# ==========================================================
# Sequence Length Analysis
# ==========================================================

logger.info("=" * 70)
logger.info("SEQUENCE LENGTH ANALYSIS")
logger.info("=" * 70)

afrihate_master["word_count"] = (

    afrihate_master["clean_text"]

    .str.split()

    .str.len()

)

print("\nWord Count Statistics\n")

print(

    afrihate_master["word_count"]

    .describe()

    .round(2)

)

percentiles = {

    "90th": np.percentile(
        afrihate_master["word_count"],
        90
    ),

    "95th": np.percentile(
        afrihate_master["word_count"],
        95
    ),

    "99th": np.percentile(
        afrihate_master["word_count"],
        99
    ),

}

print("\nSequence Length Percentiles")

for name, value in percentiles.items():

    print(
        f"{name:<5}: {int(value)} words"
    )

recommended = int(percentiles["95th"] * 1.5)

if recommended <= 64:
    max_length = 64
elif recommended <= 128:
    max_length = 128
else:
    max_length = 256

logger.info(
    f"Recommended tokenizer max_length: {max_length}"
)

# ==========================================================
# Before / After Examples
# ==========================================================

logger.info("=" * 70)
logger.info("PREPROCESSING EXAMPLES")
logger.info("=" * 70)

sample_size = min(3, len(afrihate_master))

samples = afrihate_master.sample(
    sample_size,
    random_state=RANDOM_STATE
)

for i, (_, row) in enumerate(samples.iterrows(), start=1):

    print(f"\nExample {i}")

    print("-" * 60)

    print("RAW")

    print(row["text"])

    print()

    print("CLEAN")

    print(row["clean_text"])

# ==========================================================
# Cleaning Summary
# ==========================================================

logger.info("=" * 70)
logger.info("SECTION 2 COMPLETE")
logger.info("=" * 70)

logger.info(
    f"Clean records available: "
    f"{len(afrihate_master):,}"
)

logger.info(
    "Dataset ready for label harmonisation."
)

In [ ]:
# ==========================================================
# Section 3A: Label Harmonisation & Cross-Dataset Integrity
# ==========================================================
#
# Purpose
# -------
# Harmonise AfriHate labels and identify any records that
# overlap with the frozen PolitikWeli datasets.
#
# This section:
#   • Standardises labels
#   • Creates a strict binary hate target
#   • Detects cross-dataset text overlap
#   • Removes overlapping records
#
# Output
# ------
# afrihate_master (overlap-free)
#
# ==========================================================

logger.info("=" * 70)
logger.info("LABEL HARMONISATION")
logger.info("=" * 70)

# ==========================================================
# Label Mapping
# ==========================================================

LABEL_TO_ID = {

    "normal": 0,

    "abusive": 1,

    "abuse": 1,

    "hate": 2,

    "hate speech": 2,

}

ID_TO_LABEL = {

    0: "normal",

    1: "abusive",

    2: "hate",

}

logger.info("Standardising class labels...")

# ----------------------------------------------------------
# Normalise label strings
# ----------------------------------------------------------

afrihate_master["label"] = (

    afrihate_master["label"]

    .astype(str)

    .str.lower()

    .str.strip()

)

# ----------------------------------------------------------
# Map labels
# ----------------------------------------------------------

afrihate_master["label_id"] = (

    afrihate_master["label"]

    .map(LABEL_TO_ID)

)

# ----------------------------------------------------------
# Validate mapping
# ----------------------------------------------------------

unknown_labels = (

    afrihate_master[
        afrihate_master["label_id"].isna()
    ]["label"]

    .unique()

)

if len(unknown_labels) > 0:

    raise ValueError(

        "Unknown labels detected:\n"

        f"{unknown_labels}"

    )

afrihate_master["label_id"] = (

    afrihate_master["label_id"]

    .astype(int)

)

afrihate_master["label_text"] = (

    afrihate_master["label_id"]

    .map(ID_TO_LABEL)

)

logger.info("✓ Labels harmonised.")

print("\nClass Distribution\n")

print(

    afrihate_master["label_text"]

    .value_counts()

)

# ==========================================================
# Create Binary Hate Target
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING BINARY HATE TARGET")
logger.info("=" * 70)

# Binary target used only for optional analyses.
#
# normal  -> 0
# abusive -> 0
# hate    -> 1

afrihate_master["is_strict_hate"] = (

    afrihate_master["label_id"] == 2

).astype(int)

print("\nBinary Hate Distribution\n")

print(

    afrihate_master["is_strict_hate"]

    .value_counts()

)

# ==========================================================
# Helper Function
# ==========================================================

def find_text_overlap(source_df, reference_df, source_name):
    """
    Return texts in source_df that also occur in reference_df.
    """

    overlap = (

        source_df["clean_text"]

        .isin(reference_df["text"])

    )

    overlap_df = source_df.loc[overlap].copy()

    logger.info(

        f"{source_name:<15}: "

        f"{len(overlap_df):>5,} overlaps"

    )

    return overlap_df

# ==========================================================
# Cross-Dataset Integrity Check
# ==========================================================

logger.info("=" * 70)
logger.info("CHECKING CROSS-DATASET OVERLAP")
logger.info("=" * 70)

master_overlap = find_text_overlap(

    afrihate_master,

    politikweli_master,

    "Master"

)

train_overlap = find_text_overlap(

    afrihate_master,

    politikweli_train,

    "Train"

)

validation_overlap = find_text_overlap(

    afrihate_master,

    politikweli_validation,

    "Validation"

)

test_overlap = find_text_overlap(

    afrihate_master,

    politikweli_test,

    "Test"

)

# ==========================================================
# Remove Overlapping Records
# ==========================================================

logger.info("=" * 70)
logger.info("REMOVING OVERLAPPING RECORDS")
logger.info("=" * 70)

texts_to_remove = set(

    master_overlap["clean_text"]

)

logger.info(

    f"Unique overlapping texts: "

    f"{len(texts_to_remove):,}"

)

before = len(afrihate_master)

afrihate_master = (

    afrihate_master[
        ~afrihate_master["clean_text"]

        .isin(texts_to_remove)
    ]

    .copy()

)

removed = before - len(afrihate_master)

logger.info(

    f"Removed {removed:,} overlapping records."

)

logger.info(

    f"Remaining records: "

    f"{len(afrihate_master):,}"

)

logger.info("Section 3A completed successfully.")

In [ ]:
# ==========================================================
# Section 3B: Conflict Resolution & Dataset Validation
# ==========================================================
#
# Purpose
# -------
# Finalise the harmonised AfriHate dataset by removing
# conflicting labels, eliminating duplicate records and
# validating dataset integrity.
#
# This section:
#   • Removes conflicting texts
#   • Removes exact duplicates
#   • Performs integrity validation
#   • Produces summary statistics
#
# Output
# ------
# afrihate_master (harmonised)
#
# ==========================================================

logger.info("=" * 70)
logger.info("CONFLICT RESOLUTION")
logger.info("=" * 70)

# ==========================================================
# Remove Conflicting Labels
# ==========================================================
#
# A conflict exists when identical cleaned text appears
# with more than one class label.
#

label_counts = (

    afrihate_master

    .groupby("clean_text")["label_text"]

    .nunique()

)

conflicting_texts = label_counts[

    label_counts > 1

].index

logger.info(

    f"Conflicting texts detected: "

    f"{len(conflicting_texts):,}"

)

before = len(afrihate_master)

afrihate_master = (

    afrihate_master[
        ~afrihate_master["clean_text"].isin(
            conflicting_texts
        )
    ]

    .copy()

)

removed_conflicts = before - len(afrihate_master)

logger.info(

    f"Removed {removed_conflicts:,} "

    f"records with conflicting labels."

)

# ==========================================================
# Remove Exact Duplicates
# ==========================================================

logger.info("=" * 70)
logger.info("REMOVING DUPLICATE RECORDS")
logger.info("=" * 70)

before = len(afrihate_master)

afrihate_master = (

    afrihate_master

    .drop_duplicates(

        subset=[

            "clean_text",

            "label_text"

        ],

        keep="first"

    )

    .copy()

)

duplicates_removed = before - len(afrihate_master)

logger.info(

    f"Removed {duplicates_removed:,} "

    f"duplicate records."

)

# ==========================================================
# Remove Duplicate IDs (Safety Check)
# ==========================================================

if "tweet_id" in afrihate_master.columns:

    before = len(afrihate_master)

    afrihate_master = (

        afrihate_master

        .drop_duplicates(

            subset="tweet_id",

            keep="first"

        )

        .copy()

    )

    duplicate_ids = before - len(afrihate_master)

    logger.info(

        f"Removed {duplicate_ids:,} "

        f"duplicate tweet IDs."

    )

# ==========================================================
# Dataset Integrity Validation
# ==========================================================

logger.info("=" * 70)
logger.info("VALIDATING DATASET")
logger.info("=" * 70)

assert (

    afrihate_master["clean_text"]

    .isna()

    .sum()

    == 0

), "Missing cleaned text detected."

assert (

    afrihate_master["label_id"]

    .isna()

    .sum()

    == 0

), "Missing labels detected."

assert (

    afrihate_master["clean_text"]

    .duplicated()

    .sum()

    == 0

), "Duplicate texts remain."

assert (

    afrihate_master["label_id"]

    .isin([0, 1, 2])

    .all()

), "Unexpected label IDs detected."

logger.info("✓ Dataset integrity checks passed.")

# ==========================================================
# Final Dataset Statistics
# ==========================================================

logger.info("=" * 70)
logger.info("FINAL DATASET SUMMARY")
logger.info("=" * 70)

logger.info(

    f"Final dataset size: "

    f"{len(afrihate_master):,}"

)

print("\nThree-Class Distribution\n")

print(

    afrihate_master["label_text"]

    .value_counts()

    .sort_index()

)

print("\nThree-Class Percentages\n")

print(

    (

        afrihate_master["label_text"]

        .value_counts(normalize=True)

        * 100

    )

    .round(2)

    .sort_index()

)

print("\nBinary Hate Distribution\n")

print(

    afrihate_master["is_strict_hate"]

    .value_counts()

    .sort_index()

)

# ==========================================================
# Cross-Dataset Verification
# ==========================================================

logger.info("=" * 70)
logger.info("FINAL OVERLAP VERIFICATION")
logger.info("=" * 70)

remaining_overlap = (

    afrihate_master["clean_text"]

    .isin(

        politikweli_master["text"]

    )

    .sum()

)

assert (

    remaining_overlap == 0

), "Overlap with PolitikWeli still exists."

logger.info(

    "✓ No remaining overlap with PolitikWeli."

)

# ==========================================================
# Preview Harmonised Dataset
# ==========================================================

logger.info("=" * 70)
logger.info("DATASET PREVIEW")
logger.info("=" * 70)

preview_columns = [

    col for col in [

        "clean_text",

        "label_text",

        "label_id",

        "is_strict_hate",

        "hf_split"

    ]

    if col in afrihate_master.columns

]

print(

    afrihate_master[preview_columns]

    .head()

)

logger.info("=" * 70)
logger.info("SECTION 3 COMPLETE")
logger.info("=" * 70)

logger.info(

    "AfriHate dataset successfully harmonised "

    "and validated."

)

In [ ]:
# ==========================================================
# Section 4A: Dataset Profiling & Frozen Dataset Splits
# ==========================================================
#
# Purpose
# -------
# Finalise the harmonised AfriHate dataset by profiling the
# dataset, creating reproducible train/validation/test splits,
# and verifying that the splits preserve the class
# distribution without introducing data leakage.
#
# Outputs (intermediate)
# ----------------------
# afrihate_train_df
# afrihate_validation_df
# afrihate_test_df
#
# ==========================================================

import json
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

logger.info("=" * 70)
logger.info("FINAL DATASET PROFILING")
logger.info("=" * 70)

# ==========================================================
# Dataset Source Profile
# ==========================================================

logger.info("Dataset source profile")

if "hf_split" in afrihate_master.columns:

    split_distribution = (

        afrihate_master["hf_split"]
        .value_counts()
        .sort_index()

    )

    print("\nOriginal Hugging Face Split Distribution\n")

    print(split_distribution)

# ==========================================================
# Final Class Distribution
# ==========================================================

distribution = pd.DataFrame({

    "Count":

        afrihate_master["label_text"]
        .value_counts()
        .sort_index(),

    "Percentage (%)":

        (
            afrihate_master["label_text"]
            .value_counts(normalize=True)
            .sort_index()
            * 100
        ).round(2)

})

print("\nFinal Class Distribution\n")

print(distribution)

# ==========================================================
# Create Frozen Train / Validation / Test Splits
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING FROZEN DATASET SPLITS")
logger.info("=" * 70)

#
# 80% Training
# 10% Validation
# 10% Test
#
# Stratification is performed using the original
# three-class label to preserve the class distribution.
#

afrihate_train_df, temp_df = train_test_split(

    afrihate_master,

    test_size=0.20,

    stratify=afrihate_master["label_id"],

    random_state=RANDOM_STATE,

)

afrihate_validation_df, afrihate_test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label_id"],

    random_state=RANDOM_STATE,

)

logger.info(
    f"Train samples      : {len(afrihate_train_df):,}"
)

logger.info(
    f"Validation samples : {len(afrihate_validation_df):,}"
)

logger.info(
    f"Test samples       : {len(afrihate_test_df):,}"
)

# ==========================================================
# Verify Split Integrity
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING SPLIT INTEGRITY")
logger.info("=" * 70)

verify_no_overlap(

    afrihate_train_df,

    afrihate_validation_df,

    afrihate_test_df,

    text_column="clean_text"

)

logger.info("✓ No train/validation overlap detected.")
logger.info("✓ No train/test overlap detected.")
logger.info("✓ No validation/test overlap detected.")

# ==========================================================
# Verify Class Distribution
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING CLASS DISTRIBUTIONS")
logger.info("=" * 70)

split_distribution = pd.DataFrame({

    "Overall %":

        (
            afrihate_master["label_text"]
            .value_counts(normalize=True)
            * 100
        ),

    "Train %":

        (
            afrihate_train_df["label_text"]
            .value_counts(normalize=True)
            * 100
        ),

    "Validation %":

        (
            afrihate_validation_df["label_text"]
            .value_counts(normalize=True)
            * 100
        ),

    "Test %":

        (
            afrihate_test_df["label_text"]
            .value_counts(normalize=True)
            * 100
        )

}).round(2)

print("\nClass Distribution Across Splits\n")

print(split_distribution.fillna(0))

# ==========================================================
# Verify Binary Hate Target
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING BINARY TARGET")
logger.info("=" * 70)

for name, df in {

    "Train": afrihate_train_df,

    "Validation": afrihate_validation_df,

    "Test": afrihate_test_df

}.items():

    expected_binary = (

        df["label_id"] == 2

    ).astype(int)

    if not expected_binary.equals(

        df["is_strict_hate"]

    ):

        raise ValueError(

            f"{name}: binary target verification failed."

        )

logger.info("✓ Binary target verified across all splits.")

logger.info("=" * 70)
logger.info("SECTION 4A COMPLETE")
logger.info("=" * 70)

logger.info(
    "Proceeding to dataset export and class-weight computation."
)

In [ ]:
# ==========================================================
# Section 4B: Compute Training Class Weights & Export Datasets
# ==========================================================
#
# Purpose
# -------
# Compute class weights from the training split only and
# export the frozen AfriHate datasets for downstream use.
#
# Outputs
# -------
# data/processed/afrihate/
#     afrihate_master.csv
#     afrihate_train.csv
#     afrihate_validation.csv
#     afrihate_test.csv
#     afrihate_dataset_summary.csv
#     afrihate_split_summary.csv
#
# ==========================================================

logger.info("=" * 70)
logger.info("COMPUTING TRAINING CLASS WEIGHTS")
logger.info("=" * 70)

# ==========================================================
# Compute Three-Class Training Weights
# ==========================================================

#
# IMPORTANT
# ---------
# Class weights are computed ONLY from the training set.
# This avoids incorporating information from the validation
# or test sets when training the model.
#

weights = compute_class_weight(

    class_weight="balanced",

    classes=np.array([0, 1, 2]),

    y=afrihate_train_df["label_id"]

)

class_weights = {

    "normal": float(weights[0]),

    "abusive": float(weights[1]),

    "hate": float(weights[2])

}

print("\nTraining Class Weights\n")

for key, value in class_weights.items():

    print(f"{key:<10}: {value:.4f}")

# ==========================================================
# Export Frozen Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("EXPORTING FROZEN DATASETS")
logger.info("=" * 70)

master_path = (
    PROCESSED_DIR /
    "afrihate_master.csv"
)

train_path = (
    PROCESSED_DIR /
    "afrihate_train.csv"
)

validation_path = (
    PROCESSED_DIR /
    "afrihate_validation.csv"
)

test_path = (
    PROCESSED_DIR /
    "afrihate_test.csv"
)

afrihate_master.to_csv(
    master_path,
    index=False
)

afrihate_train_df.to_csv(
    train_path,
    index=False
)

afrihate_validation_df.to_csv(
    validation_path,
    index=False
)

afrihate_test_df.to_csv(
    test_path,
    index=False
)

logger.info("✓ Frozen datasets exported successfully.")

# ==========================================================
# Create Dataset Summary
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING DATASET SUMMARY")
logger.info("=" * 70)

summary = pd.DataFrame({

    "Statistic": [

        "Total Samples",

        "Training Samples",

        "Validation Samples",

        "Test Samples",

        "Normal",

        "Abusive",

        "Hate"

    ],

    "Value": [

        len(afrihate_master),

        len(afrihate_train_df),

        len(afrihate_validation_df),

        len(afrihate_test_df),

        (afrihate_master["label_id"] == 0).sum(),

        (afrihate_master["label_id"] == 1).sum(),

        (afrihate_master["label_id"] == 2).sum()

    ]

})

summary_path = (

    PROCESSED_DIR /

    "afrihate_dataset_summary.csv"

)

summary.to_csv(

    summary_path,

    index=False

)

logger.info("✓ Dataset summary exported.")

# ==========================================================
# Create Split Summary
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING SPLIT SUMMARY")
logger.info("=" * 70)

split_summary = pd.DataFrame({

    "Split": [

        "Train",

        "Validation",

        "Test"

    ],

    "Samples": [

        len(afrihate_train_df),

        len(afrihate_validation_df),

        len(afrihate_test_df)

    ],

    "Percentage": [

        len(afrihate_train_df) / len(afrihate_master) * 100,

        len(afrihate_validation_df) / len(afrihate_master) * 100,

        len(afrihate_test_df) / len(afrihate_master) * 100

    ]

})

split_summary_path = (

    PROCESSED_DIR /

    "afrihate_split_summary.csv"

)

split_summary.to_csv(

    split_summary_path,

    index=False

)

logger.info("✓ Split summary exported.")

# ==========================================================
# Preview Exported Split Sizes
# ==========================================================

logger.info("=" * 70)
logger.info("FROZEN DATASET SUMMARY")
logger.info("=" * 70)

print(split_summary)

logger.info("=" * 70)
logger.info("SECTION 4B COMPLETE")
logger.info("=" * 70)

logger.info(

    "Proceeding to metadata and audit generation."

)

In [ ]:
# ==========================================================
# Section 4B: Compute Training Class Weights & Export Datasets
# ==========================================================
#
# Purpose
# -------
# Compute class weights from the training split only and
# export the frozen AfriHate datasets for downstream use.
#
# Outputs
# -------
# data/processed/afrihate/
#     afrihate_master.csv
#     afrihate_train.csv
#     afrihate_validation.csv
#     afrihate_test.csv
#     afrihate_dataset_summary.csv
#     afrihate_split_summary.csv
#
# ==========================================================

logger.info("=" * 70)
logger.info("COMPUTING TRAINING CLASS WEIGHTS")
logger.info("=" * 70)

# ==========================================================
# Compute Three-Class Training Weights
# ==========================================================

#
# IMPORTANT
# ---------
# Class weights are computed ONLY from the training set.
# This avoids incorporating information from the validation
# or test sets when training the model.
#

weights = compute_class_weight(

    class_weight="balanced",

    classes=np.array([0, 1, 2]),

    y=afrihate_train_df["label_id"]

)

class_weights = {

    "normal": float(weights[0]),

    "abusive": float(weights[1]),

    "hate": float(weights[2])

}

print("\nTraining Class Weights\n")

for key, value in class_weights.items():

    print(f"{key:<10}: {value:.4f}")

# ==========================================================
# Export Frozen Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("EXPORTING FROZEN DATASETS")
logger.info("=" * 70)

master_path = (
    PROCESSED_DIR /
    "afrihate_master.csv"
)

train_path = (
    PROCESSED_DIR /
    "afrihate_train.csv"
)

validation_path = (
    PROCESSED_DIR /
    "afrihate_validation.csv"
)

test_path = (
    PROCESSED_DIR /
    "afrihate_test.csv"
)

afrihate_master.to_csv(
    master_path,
    index=False
)

afrihate_train_df.to_csv(
    train_path,
    index=False
)

afrihate_validation_df.to_csv(
    validation_path,
    index=False
)

afrihate_test_df.to_csv(
    test_path,
    index=False
)

logger.info("✓ Frozen datasets exported successfully.")

# ==========================================================
# Create Dataset Summary
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING DATASET SUMMARY")
logger.info("=" * 70)

summary = pd.DataFrame({

    "Statistic": [

        "Total Samples",

        "Training Samples",

        "Validation Samples",

        "Test Samples",

        "Normal",

        "Abusive",

        "Hate"

    ],

    "Value": [

        len(afrihate_master),

        len(afrihate_train_df),

        len(afrihate_validation_df),

        len(afrihate_test_df),

        (afrihate_master["label_id"] == 0).sum(),

        (afrihate_master["label_id"] == 1).sum(),

        (afrihate_master["label_id"] == 2).sum()

    ]

})

summary_path = (

    PROCESSED_DIR /

    "afrihate_dataset_summary.csv"

)

summary.to_csv(

    summary_path,

    index=False

)

logger.info("✓ Dataset summary exported.")

# ==========================================================
# Create Split Summary
# ==========================================================

logger.info("=" * 70)
logger.info("CREATING SPLIT SUMMARY")
logger.info("=" * 70)

split_summary = pd.DataFrame({

    "Split": [

        "Train",

        "Validation",

        "Test"

    ],

    "Samples": [

        len(afrihate_train_df),

        len(afrihate_validation_df),

        len(afrihate_test_df)

    ],

    "Percentage": [

        len(afrihate_train_df) / len(afrihate_master) * 100,

        len(afrihate_validation_df) / len(afrihate_master) * 100,

        len(afrihate_test_df) / len(afrihate_master) * 100

    ]

})

split_summary_path = (

    PROCESSED_DIR /

    "afrihate_split_summary.csv"

)

split_summary.to_csv(

    split_summary_path,

    index=False

)

logger.info("✓ Split summary exported.")

# ==========================================================
# Preview Exported Split Sizes
# ==========================================================

logger.info("=" * 70)
logger.info("FROZEN DATASET SUMMARY")
logger.info("=" * 70)

print(split_summary)

logger.info("=" * 70)
logger.info("SECTION 4B COMPLETE")
logger.info("=" * 70)

logger.info(

    "Proceeding to metadata and audit generation."

)

In [ ]:
# ==========================================================
# Section 4C: Metadata & Audit Generation
# ==========================================================
#
# Purpose
# -------
# Save reproducibility metadata, class weights, and a
# comprehensive audit report describing the final frozen
# AfriHate dataset.
#
# Outputs
# -------
# data/processed/afrihate/
#     afrihate_class_weights.json
#     afrihate_metadata.json
#     afrihate_audit.json
#
# ==========================================================

logger.info("=" * 70)
logger.info("GENERATING METADATA & AUDIT REPORTS")
logger.info("=" * 70)

# ==========================================================
# Save Training Class Weights
# ==========================================================

logger.info("Saving class weights...")

class_weights_path = (

    PROCESSED_DIR /

    "afrihate_class_weights.json"

)

with open(

    class_weights_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        class_weights,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("✓ Class weights saved.")

# ==========================================================
# Save Preparation Metadata
# ==========================================================

logger.info("Saving preparation metadata...")

metadata = {

    "dataset": "AfriHate",

    "language": "Swahili",

    "source": "Hugging Face",

    "repository": "afrihate/afrihate",

    "random_state": RANDOM_STATE,

    "records": int(len(afrihate_master)),

    "train_size": int(len(afrihate_train_df)),

    "validation_size": int(len(afrihate_validation_df)),

    "test_size": int(len(afrihate_test_df)),

    "train_percentage": round(

        len(afrihate_train_df) / len(afrihate_master) * 100,

        2

    ),

    "validation_percentage": round(

        len(afrihate_validation_df) / len(afrihate_master) * 100,

        2

    ),

    "test_percentage": round(

        len(afrihate_test_df) / len(afrihate_master) * 100,

        2

    ),

    "split_strategy": "80/10/10",

    "stratification": "label_id",

    "random_seed": RANDOM_STATE,

    "text_column": "clean_text",

    "label_column": "label_id",

    "binary_target_column": "is_strict_hate",

    "labels": {

        "0": "normal",

        "1": "abusive",

        "2": "hate"

    },

    "binary_target": {

        "0": "normal_or_abusive",

        "1": "hate"

    },

    "class_weights_source": "training_split"

}

metadata_path = (

    PROCESSED_DIR /

    "afrihate_metadata.json"

)

with open(

    metadata_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        metadata,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("✓ Preparation metadata saved.")

# ==========================================================
# Build Audit Report
# ==========================================================

logger.info("Generating audit report...")

audit = {

    "dataset_statistics": {

        "total_records": int(len(afrihate_master)),

        "unique_texts": int(

            afrihate_master["clean_text"]

            .nunique()

        ),

        "duplicate_texts": int(

            afrihate_master["clean_text"]

            .duplicated()

            .sum()

        ),

        "missing_text": int(

            afrihate_master["clean_text"]

            .isna()

            .sum()

        ),

        "missing_labels": int(

            afrihate_master["label_id"]

            .isna()

            .sum()

        )

    },

    "split_statistics": {

        "train": int(len(afrihate_train_df)),

        "validation": int(len(afrihate_validation_df)),

        "test": int(len(afrihate_test_df))

    },

    "overall_class_distribution":

        afrihate_master["label_text"]

        .value_counts()

        .to_dict(),

    "training_class_distribution":

        afrihate_train_df["label_text"]

        .value_counts()

        .to_dict(),

    "validation_class_distribution":

        afrihate_validation_df["label_text"]

        .value_counts()

        .to_dict(),

    "test_class_distribution":

        afrihate_test_df["label_text"]

        .value_counts()

        .to_dict(),

    "binary_target_distribution":

        afrihate_master["is_strict_hate"]

        .value_counts()

        .to_dict(),

    "quality_checks": {

        "duplicate_texts_removed": True,

        "missing_labels_removed": True,

        "missing_text_removed": True,

        "split_leakage_detected": False,

        "binary_target_verified": True

    }

}

audit_path = (

    PROCESSED_DIR /

    "afrihate_audit.json"

)

with open(

    audit_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        audit,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("✓ Audit report saved.")

# ==========================================================
# Preview Metadata
# ==========================================================

logger.info("=" * 70)
logger.info("PREPARATION SUMMARY")
logger.info("=" * 70)

print("\nDataset")

print(f"  Name              : {metadata['dataset']}")

print(f"  Records           : {metadata['records']:,}")

print(f"  Random Seed       : {metadata['random_seed']}")

print(f"  Stratification    : {metadata['stratification']}")

print()

print("Frozen Dataset Sizes")

print(f"  Train             : {metadata['train_size']:,}")

print(f"  Validation        : {metadata['validation_size']:,}")

print(f"  Test              : {metadata['test_size']:,}")

print()

print("Metadata Files")

print(f"✓ {class_weights_path.name}")

print(f"✓ {metadata_path.name}")

print(f"✓ {audit_path.name}")

logger.info("=" * 70)
logger.info("SECTION 4C COMPLETE")
logger.info("=" * 70)

logger.info(

    "Proceeding to final validation and completion summary."

)

In [ ]:
# ==========================================================
# Section 4C: Metadata & Audit Generation
# ==========================================================
#
# Purpose
# -------
# Save reproducibility metadata, class weights, and a
# comprehensive audit report describing the final frozen
# AfriHate dataset.
#
# Outputs
# -------
# data/processed/afrihate/
#     afrihate_class_weights.json
#     afrihate_metadata.json
#     afrihate_audit.json
#
# ==========================================================

logger.info("=" * 70)
logger.info("GENERATING METADATA & AUDIT REPORTS")
logger.info("=" * 70)

# ==========================================================
# Save Training Class Weights
# ==========================================================

logger.info("Saving class weights...")

class_weights_path = (

    PROCESSED_DIR /

    "afrihate_class_weights.json"

)

with open(

    class_weights_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        class_weights,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("✓ Class weights saved.")

# ==========================================================
# Save Preparation Metadata
# ==========================================================

logger.info("Saving preparation metadata...")

metadata = {

    "dataset": "AfriHate",

    "language": "Swahili",

    "source": "Hugging Face",

    "repository": "afrihate/afrihate",

    "random_state": RANDOM_STATE,

    "records": int(len(afrihate_master)),

    "train_size": int(len(afrihate_train_df)),

    "validation_size": int(len(afrihate_validation_df)),

    "test_size": int(len(afrihate_test_df)),

    "train_percentage": round(

        len(afrihate_train_df) / len(afrihate_master) * 100,

        2

    ),

    "validation_percentage": round(

        len(afrihate_validation_df) / len(afrihate_master) * 100,

        2

    ),

    "test_percentage": round(

        len(afrihate_test_df) / len(afrihate_master) * 100,

        2

    ),

    "split_strategy": "80/10/10",

    "stratification": "label_id",

    "random_seed": RANDOM_STATE,

    "text_column": "clean_text",

    "label_column": "label_id",

    "binary_target_column": "is_strict_hate",

    "labels": {

        "0": "normal",

        "1": "abusive",

        "2": "hate"

    },

    "binary_target": {

        "0": "normal_or_abusive",

        "1": "hate"

    },

    "class_weights_source": "training_split"

}

metadata_path = (

    PROCESSED_DIR /

    "afrihate_metadata.json"

)

with open(

    metadata_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        metadata,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("✓ Preparation metadata saved.")

# ==========================================================
# Build Audit Report
# ==========================================================

logger.info("Generating audit report...")

audit = {

    "dataset_statistics": {

        "total_records": int(len(afrihate_master)),

        "unique_texts": int(

            afrihate_master["clean_text"]

            .nunique()

        ),

        "duplicate_texts": int(

            afrihate_master["clean_text"]

            .duplicated()

            .sum()

        ),

        "missing_text": int(

            afrihate_master["clean_text"]

            .isna()

            .sum()

        ),

        "missing_labels": int(

            afrihate_master["label_id"]

            .isna()

            .sum()

        )

    },

    "split_statistics": {

        "train": int(len(afrihate_train_df)),

        "validation": int(len(afrihate_validation_df)),

        "test": int(len(afrihate_test_df))

    },

    "overall_class_distribution":

        afrihate_master["label_text"]

        .value_counts()

        .to_dict(),

    "training_class_distribution":

        afrihate_train_df["label_text"]

        .value_counts()

        .to_dict(),

    "validation_class_distribution":

        afrihate_validation_df["label_text"]

        .value_counts()

        .to_dict(),

    "test_class_distribution":

        afrihate_test_df["label_text"]

        .value_counts()

        .to_dict(),

    "binary_target_distribution":

        afrihate_master["is_strict_hate"]

        .value_counts()

        .to_dict(),

    "quality_checks": {

        "duplicate_texts_removed": True,

        "missing_labels_removed": True,

        "missing_text_removed": True,

        "split_leakage_detected": False,

        "binary_target_verified": True

    }

}

audit_path = (

    PROCESSED_DIR /

    "afrihate_audit.json"

)

with open(

    audit_path,

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        audit,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("✓ Audit report saved.")

# ==========================================================
# Preview Metadata
# ==========================================================

logger.info("=" * 70)
logger.info("PREPARATION SUMMARY")
logger.info("=" * 70)

print("\nDataset")

print(f"  Name              : {metadata['dataset']}")

print(f"  Records           : {metadata['records']:,}")

print(f"  Random Seed       : {metadata['random_seed']}")

print(f"  Stratification    : {metadata['stratification']}")

print()

print("Frozen Dataset Sizes")

print(f"  Train             : {metadata['train_size']:,}")

print(f"  Validation        : {metadata['validation_size']:,}")

print(f"  Test              : {metadata['test_size']:,}")

print()

print("Metadata Files")

print(f"✓ {class_weights_path.name}")

print(f"✓ {metadata_path.name}")

print(f"✓ {audit_path.name}")

logger.info("=" * 70)
logger.info("SECTION 4C COMPLETE")
logger.info("=" * 70)

logger.info(

    "Proceeding to final validation and completion summary."

)